## Importing libraries

In [12]:
import pandas as pd
import numpy as np
import re

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

import ssl
ssl._create_default_https_context = ssl._create_unverified_context

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/msvmilind/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /Users/msvmilind/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/msvmilind/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/msvmilind/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

## Data Loading

In [3]:
# Loading dataset and checking data
df = pd.read_csv("IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
# Checking size of data
df.shape

(50000, 2)

In [5]:
# Checking for null values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   review     50000 non-null  str  
 1   sentiment  50000 non-null  str  
dtypes: str(2)
memory usage: 781.4 KB


In [6]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [7]:
df.describe

<bound method NDFrame.describe of                                                   review sentiment
0      One of the other reviewers has mentioned that ...  positive
1      A wonderful little production. <br /><br />The...  positive
2      I thought this was a wonderful way to spend ti...  positive
3      Basically there's a family where a little boy ...  negative
4      Petter Mattei's "Love in the Time of Money" is...  positive
...                                                  ...       ...
49995  I thought this movie did a down right good job...  positive
49996  Bad plot, bad dialogue, bad acting, idiotic di...  negative
49997  I am a Catholic taught in parochial elementary...  negative
49998  I'm going to have to disagree with the previou...  negative
49999  No one expects the Star Trek movies to be high...  negative

[50000 rows x 2 columns]>

### Fucntion

In [15]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    
    # removing URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # removing HTML tags 
    text = re.sub(r'<.*?>', '', text)

    # converting to lowercase
    text = text.lower()

    # removing punctuation and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # tokeinzing
    tokens = word_tokenize(text)

    # removing stopwords and lemmatizing
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]

    return " ".join(tokens)


## Preprocessing

In [16]:
df['cleaned_review'] = df['review'].apply(preprocess_text)

In [19]:
df[['cleaned_review', 'review']].head()

,cleaned_review,review
0,one reviewer mentioned watching oz episode you...,One of the other reviewers has mentioned that ...
1,wonderful little production filming technique ...,A wonderful little production. <br /><br />The...
2,thought wonderful way spend time hot summer we...,I thought this was a wonderful way to spend ti...
3,basically there family little boy jake think t...,Basically there's a family where a little boy ...
4,petter matteis love time money visually stunni...,"Petter Mattei's ""Love in the Time of Money"" is..."


In [20]:
df.describe

<bound method NDFrame.describe of                                                   review sentiment  \
0      One of the other reviewers has mentioned that ...  positive   
1      A wonderful little production. <br /><br />The...  positive   
2      I thought this was a wonderful way to spend ti...  positive   
3      Basically there's a family where a little boy ...  negative   
4      Petter Mattei's "Love in the Time of Money" is...  positive   
...                                                  ...       ...   
49995  I thought this movie did a down right good job...  positive   
49996  Bad plot, bad dialogue, bad acting, idiotic di...  negative   
49997  I am a Catholic taught in parochial elementary...  negative   
49998  I'm going to have to disagree with the previou...  negative   
49999  No one expects the Star Trek movies to be high...  negative   

                                          cleaned_review  
0      one reviewer mentioned watching oz episode you...  
1      

In [21]:
# ML utilities
from sklearn.model_selection import train_test_split

# Feature engineering
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

# Evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [23]:
from sklearn.model_selection import train_test_split
X = df['cleaned_review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Feature Engineering

In [ ]:
# Bag of Words
bow = CountVectorizer()

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

In [25]:
# TF-IDF
tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

## Models

### Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_tfidf, y_train)
lr_pred = lr.predict(X_test_tfidf)

### Naive Bayes

In [27]:
nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)
nb_pred = nb.predict(X_test_tfidf)

### Deicsion Tree

In [28]:
dt = DecisionTreeClassifier()
dt.fit(X_train_tfidf, y_train)
dt_pred = dt.predict(X_test_tfidf)

## Evaluation

In [29]:
def evaluate_model(y_test, y_pred, model_name):

    print(f"\n{model_name} Results:")
    print("Accuracy:", accuracy_score(y_test,y_pred))
    print("Precision:", precision_score(y_test,y_pred, pos_label='positive'))
    print("Recall:", recall_score(y_test,y_pred, pos_label='positive'))
    print("F1:", f1_score(y_test,y_pred, pos_label='positive'))

    print("\nClassification Summary:")
    print(classification_report(y_test,y_pred))

In [30]:
evaluate_model(y_test, lr_pred, "Logistic Regression")
evaluate_model(y_test, nb_pred, "Naive Bayes")
evaluate_model(y_test, dt_pred, "Decision Tree")


Logistic Regression Results:
Accuracy: 0.8942
Precision: 0.8844890863434421
Recall: 0.9087120460408811
F1: 0.8964369616288176

Classification Summary:
              precision    recall  f1-score   support

    negative       0.90      0.88      0.89      4961
    positive       0.88      0.91      0.90      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000


Naive Bayes Results:
Accuracy: 0.8666
Precision: 0.8790669122160835
Recall: 0.8525501091486406
F1: 0.8656054805561153

Classification Summary:
              precision    recall  f1-score   support

    negative       0.85      0.88      0.87      4961
    positive       0.88      0.85      0.87      5039

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000


Decision Tree Results:
Accuracy: 0.7135
Precision

In [31]:
results = pd.DataFrame({
    'Model':['Logistic Regression','Naive Bayes','Decision Tree'],
    'Accuracy':[
        accuracy_score(y_test,lr_pred),
        accuracy_score(y_test,nb_pred),
        accuracy_score(y_test,dt_pred)
    ],
    'F1 Score':[
        f1_score(y_test,lr_pred,pos_label='positive'),
        f1_score(y_test,nb_pred,pos_label='positive'),
        f1_score(y_test,dt_pred,pos_label='positive')
    ]
})

results

,Model,Accuracy,F1 Score
0,Logistic Regression,0.8942,0.896437
1,Naive Bayes,0.8666,0.865605
2,Decision Tree,0.7135,0.717260
